In [1]:

# load packages
import refinitiv.data as rd
import numpy as np 
import eikon as ek
import pandas as pd
import datetime as dt
import pickle
import configparser as cp
from scipy.stats.mstats import winsorize

ModuleNotFoundError: No module named 'refinitiv'

In [ ]:
# date
this_year=dt.datetime.now().year

# september of 6 years ago
start=dt.datetime(this_year-6,9,1)
# today 
end=dt.datetime(this_year,8,1)

# reformat date
start=start.strftime('%Y-%m-%d')
end=end.strftime('%Y-%m-%d')

In [ ]:
# load allstock
with open('all_data.pkl','rb') as f:
    all_data=pickle.load(f)

namelist, industrylist, cusiplist, indexlist = all_data

# get RIC list which is the investment universe
RIC= cusiplist['RIC'].unique().tolist()

In [ ]:
# variables
vars = ['TR.PriceClose','TR.TotalReturn','TR.BIDPRICE',
        'TR.ASKPRICE','TR.Volume','TR.F.BookValuePctMktCap',
        'TR.CompanyMarketCapitalization','TR.NumOfStrongBuy','TR.NumOfBuy',
        'TR.NumOfStrongSell','TR.NumOfSell']

In [ ]:
# construct query list
query_list=[]
for var in vars:
    query_list.append(f'{var}(Frq=D,SDate={start},EDate={end}).date')
    query_list.append(f'{var}(Frq=D,SDate={start},EDate={end})')
query_list

In [ ]:
# I munually specified currency for some variables
query_list=['TR.PriceClose(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.PriceClose(Frq=D,SDate=2019-08-31,Curn=EUR,EDate=2025-09-20)',
 'TR.TotalReturn(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.TotalReturn(Frq=D,SDate=2019-08-31,EDate=2025-09-20)',
 'TR.BIDPRICE(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.BIDPRICE(Frq=D,SDate=2019-08-31,Curn=Native,EDate=2025-09-20)',
 'TR.ASKPRICE(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.ASKPRICE(Frq=D,SDate=2019-08-31,Curn=Native,EDate=2025-09-20)',
 'TR.Volume(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.Volume(Frq=D,SDate=2019-08-31,EDate=2025-09-20)',
 'TR.F.BookValuePctMktCap(Frq=D,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.F.BookValuePctMktCap(Frq=D,SDate=2019-08-31,EDate=2025-09-20)',
 'TR.CompanyMarketCapitalization(Frq=FY,Curn=EUR,SDate=2019-08-31,EDate=2025-09-20).date',
 'TR.CompanyMarketCapitalization(Frq=FY,Curn=EUR,SDate=2019-08-31,EDate=2025-09-20)',
 'TR.NumOfStrongBuy(Frq=D,SDate=2019-09-01,EDate=2025-09-21).date',
 'TR.NumOfStrongBuy(Frq=D,SDate=2019-09-01,EDate=2025-09-21)',
 'TR.NumOfBuy(Frq=D,SDate=2019-09-01,EDate=2025-09-21).date',
 'TR.NumOfBuy(Frq=D,SDate=2019-09-01,EDate=2025-09-21)',
 'TR.NumOfStrongSell(Frq=D,SDate=2019-09-01,EDate=2025-09-21).date',
 'TR.NumOfStrongSell(Frq=D,SDate=2019-09-01,EDate=2025-09-21)',
 'TR.NumOfSell(Frq=D,SDate=2019-09-01,EDate=2025-09-21).date',
 'TR.NumOfSell(Frq=D,SDate=2019-09-01,EDate=2025-09-21)']
 

In [ ]:
print(len(query_list))

In [ ]:
# # I encountered issues when pulling all data at once, so I split the query into several parts
# # and pull them one by one
# data_dic={}
# rd.open_session()
# data_dic[vars[0]]= rd.get_data(RIC, query_list[0:2]) # TR.PriceClose
# data_dic[vars[1]]= rd.get_data(RIC, query_list[2:4]) # TR.TotalReturn
# data_dic[vars[2]]= rd.get_data(RIC, query_list[4:6]) # TR.BIDPRICE

# rd.close_session()

In [ ]:
# rd.open_session()
# data_dic[vars[3]]= rd.get_data(RIC, query_list[6:8]) # TR.ASKPRICE
# rd.close_session()

In [ ]:
# rd.open_session()
# data_dic[vars[4]]= rd.get_data(RIC, query_list[8:10]) # TR.Volume in shares
# rd.close_session()

In [ ]:
# rd.open_session()
# data_dic[vars[5]]= rd.get_data(RIC, query_list[10:12]) # TR.F.BookValuePctMktCap
# rd.close_session()

In [ ]:
# rd.open_session()
# data_dic[vars[6]]= rd.get_data(RIC, query_list[12:14]) # TR.CompanyMarketCapitalization in EUR
# rd.close_session()

In [ ]:
# rd.open_session()
# data_dic[vars[7]]= rd.get_data(RIC, query_list[14:16]) 
# rd.close_session()

# rd.open_session()
# data_dic[vars[8]]= rd.get_data(RIC, query_list[16:18]) 
# rd.close_session()

# rd.open_session()
# data_dic[vars[9]]= rd.get_data(RIC, query_list[18:20]) 
# rd.close_session()

# rd.open_session()
# data_dic[vars[10]]= rd.get_data(RIC, query_list[20:22]) 
# rd.close_session()


In [ ]:
# for i in range(len(vars)):
#     temp=data_dic[vars[i]].copy()
#     #  drop duplicates
#     temp=temp.drop_duplicates()
#     # pivot table using Date as index and RIC as columns
#     temp.columns=['Instrument','Date','value']
#     temp=temp.pivot(index='Date',columns='Instrument',values='value')
#     data_dic[vars[i]]=temp

# # save data_dic
# with open('data_dic.pkl','wb') as f:
#     pickle.dump(data_dic,f)

In [ ]:
# load data_dic
with open('data_dic.pkl','rb') as f:
    hw2=pickle.load(f)

In [ ]:
# map dataframe columns from names to dscode
print(hw2.keys())

In [ ]:
# price 
price_local=hw2[vars[0]]

# repeat the same for other dataframes
totalret=hw2[vars[1]] # totalret
volume=hw2[vars[4]] # volume in shares
bm=hw2[vars[5]] # b/m
cap=hw2[vars[6]] # market cap
bid=hw2[vars[2]] # bid
ask=hw2[vars[3]] # ask
# stock recommendation
strongbuy=hw2[vars[7]].ffill()
strongbuy=strongbuy.bfill()
buy=hw2[vars[8]].ffill()
buy=buy.bfill()
strongsell=hw2[vars[9]].ffill()
strongsell=strongsell.bfill()
sell=hw2[vars[10]].ffill()
sell=sell.bfill()

# rec1= strongbuy+buy-strongsell-sell
rec1= strongbuy-buy-strongsell+sell

In [ ]:
# volume conversion
volume_dollar=volume*price_local

In [ ]:
# calculate spread
bid = bid.to_numpy()
ask= ask.to_numpy()
spread= (ask-bid)/(ask+bid)*2 # calculate spread as requested in the hw
spread=pd.DataFrame(spread)
spread.index=price_local.index
spread.columns=price_local.columns
spread.head()

In [ ]:
# forward fill book to market 
bm = bm.ffill()

In [ ]:
# output into pickle file
hw3={'price':price_local,
      'tri':totalret,
      'volume':volume,
      'mtbv':bm,
      'cap':cap,
      'tcost':spread,
      'rec':rec1}

In [ ]:
tri = hw3['tri'].copy()
# convert all columns to numeric, coerce errors to NaN
tri = tri.apply(pd.to_numeric, errors='coerce')
# calculate daily return, avoid division by zero
denom = tri.shift(1).replace(0, np.nan)
tri = tri / denom - 1

# for every dataframe in hw3, drop if index is NaT, then select data between start and end
for key in hw3.keys():
    if isinstance(hw3[key], pd.DataFrame):
        # Remove rows where the index is NaT
        hw3[key] = hw3[key][~hw3[key].index.isna()]
        # Select data between start and end
        hw3[key] = hw3[key].loc[(hw3[key].index >= start) & (hw3[key].index <= end)]
        



# align index of every dataframe to that of price 
for key in hw3.keys():
    if isinstance(hw3[key], pd.DataFrame):
        hw3[key] = hw3[key].reindex(hw3['price'].index)

# check data dimensions 
for key in hw3.keys():
    if isinstance(hw3[key], pd.DataFrame):
        print(f"{key}: {hw3[key].shape}")

# forward fill missing value in mtbv, then backward fill
hw3['mtbv'] = hw3['mtbv'].ffill()
        

In [ ]:
# unpakc hw3
price_local=hw3['price']
totalret=hw3['tri']
volume=hw3['volume']
bm=hw3['mtbv']
cap=hw3['cap']
spread=hw3['tcost']
rec1=hw3['rec']


In [ ]:
# creating isactive

# price condition 1 
price_active=price_local.isna().rolling(window=252).sum()
price_active=price_active<(0.1*252)

# price condition 2
price_active2= price_local.rolling(10).std()
price_active2= price_active2 != 0


# totalret
totalret_active=totalret.isna().rolling(window=252).sum()
totalret_active=totalret_active<(0.1*252)

# book value (I use bm instead of book value)
bm_active=bm.notna()

# market cap
cap_active=cap.notna()

# spread ( daily spread data are sometimes not available, such as '2020-05-01'), so I checked wether spread data is available 90% of the time in the previous month) 
spread_valid=spread.isna().rolling(window=22,min_periods=1).sum() # min_periods =1 to avoid np.nan
spread_valid=(spread_valid<=3).shift(1) # shift 1 to avoid lookahead bias

# rec 
rec_active=rec1.cumsum(axis=0)!=0


# Ensure numeric dtype for rolling operations
spread_numeric = spread.apply(pd.to_numeric, errors='coerce')
volume_dollar_numeric = volume_dollar.apply(pd.to_numeric, errors='coerce')

# further liquidity condition 1 
spread_mean = spread_numeric.rolling(window=22, min_periods=1).mean() # to address the issue that spread is not available on some days
spread_valid1 = (spread_valid.shift(1) == True) & (spread_mean < 0.01) 
spread_valid1 = spread_valid1.shift(1) # shift 1 to avoid lookahead bias
spread_valid2 = (spread_valid.shift(1) == False) & (spread_mean < 0.008)
spread_valid2 = spread_valid2.shift(1) # shift 1 to avoid lookahead bias

# volume
volume_active=volume_dollar.isna().rolling(window=252).sum()
volume_active=volume_active<(0.1*252)
# align volume active index with main active matrix index
volume_active = volume_active.loc[price_active.index]

isactive= price_active & price_active2 & totalret_active & bm_active & cap_active & spread_valid & rec_active 





In [ ]:
# # further liquidity condition 2 
volume_valid1 = (isactive.rolling(22).sum() >= (0.9 * 22)) & (volume_dollar_numeric.rolling(window=22, min_periods=1).mean() > 1e6)  # setting min period to avoid np.nan 
volume_valid1 = volume_valid1.shift(1) # shift 1 to avoid lookahead bias
volume_valid2 = (isactive.rolling(22).sum() < (0.9 * 22)) & (volume_dollar_numeric.rolling(window=22, min_periods=1).mean() > 1.2e6)
volume_valid2 = volume_valid2.shift(1) # shift 1 to avoid lookahead bias

# 
volume_enhanced = (volume_valid1 | volume_valid2)
volume_enhanced = volume_enhanced.loc[price_active.index]

In [ ]:
isactive= isactive & volume_active & volume_enhanced
num_active = isactive.sum(axis=1)
num_active

In [ ]:
# output into pickle file
hw3['isactive']=isactive

with open('hw3_input.pickle', 'wb') as handle:
      pickle.dump(hw3, handle, protocol=pickle.HIGHEST_PROTOCOL)